In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
TRAIN_DIR = "/content/drive/MyDrive/output/train"
VAL_DIR = "/content/drive/MyDrive/output/test"


In [ ]:
%%writefile config.py
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_DIR = "/content/drive/MyDrive/output/train"
VAL_DIR = "/content/drive/MyDrive/output/test"
LEARNING_RATE = 2e-4
BATCH_SIZE = 16
NUM_WORKERS = 2
IMAGE_SIZE = 256
CHANNELS_IMG = 3
L1_LAMBDA = 100
LAMBDA_GP = 10
NUM_EPOCHS = 500
LOAD_MODEL = False
SAVE_MODEL = False
CHECKPOINT_DISC = "disc.pth.tar"
CHECKPOINT_GEN = "gen.pth.tar"

both_transform = A.Compose(
    [A.Resize(width=256, height=256)],
    additional_targets={"image0": "image"},
)

transform_only_input = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(p=0.2),
        A.Normalize(mean=[0.5]*3, std=[0.5]*3, max_pixel_value=255.0),
        ToTensorV2(),
    ]
)

transform_only_mask = A.Compose(
    [
        A.Normalize(mean=[0.5]*3, std=[0.5]*3, max_pixel_value=255.0),
        ToTensorV2(),
    ]
)


Overwriting config.py


In [ ]:
%%writefile dataset.py
import numpy as np
import config
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image

class MapDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.list_files = os.listdir(self.root_dir)

    def __len__(self):
        return len(self.list_files)

    def __getitem__(self, index):
        img_file = self.list_files[index]
        img_path = os.path.join(self.root_dir, img_file)
        image = np.array(Image.open(img_path))

        # Ensure width is divisible by 2
        width = image.shape[1]
        half_width = width // 2
        input_image = image[:, :half_width, :]
        target_image = image[:, half_width:, :]

        # Albumentations
        augmentations = config.both_transform(image=input_image, image0=target_image)
        input_image = augmentations["image"]
        target_image = augmentations["image0"]

        input_image = config.transform_only_input(image=input_image)["image"]
        target_image = config.transform_only_mask(image=target_image)["image"]

        return input_image, target_image



Overwriting dataset.py


In [ ]:
%%writefile discriminator_model.py
import torch
import torch.nn as nn

class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super(CNNBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 4, stride, 1, bias=False, padding_mode="reflect"),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2),
        )

    def forward(self, x):
        return self.conv(x)

class Discriminator(nn.Module):
    def __init__(self, in_channels=3, features=[64, 128, 256, 512]):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv2d(in_channels * 2, features[0], 4, 2, 1, padding_mode="reflect"),
            nn.LeakyReLU(0.2),
        )

        layers = []
        in_channels = features[0]
        for feature in features[1:]:
            layers.append(
                CNNBlock(in_channels, feature, stride=1 if feature == features[-1] else 2)
            )
            in_channels = feature

        layers.append(
            nn.Conv2d(in_channels, 1, kernel_size=4, stride=1, padding=1, padding_mode="reflect")
        )
        self.model = nn.Sequential(*layers)

    def forward(self, x, y):
        x = torch.cat([x, y], dim=1)
        x = self.initial(x)
        return self.model(x)


Overwriting discriminator_model.py


In [ ]:
%%writefile generator_model.py
import torch
import torch.nn as nn

class Block(nn.Module):
    def __init__(self, in_channels, out_channels, down=True, act="relu", use_dropout=False):
        super(Block, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False, padding_mode="reflect")
            if down else
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU() if act == "relu" else nn.LeakyReLU(0.2),
        )
        self.use_dropout = use_dropout
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.conv(x)
        return self.dropout(x) if self.use_dropout else x

class Generator(nn.Module):
    def __init__(self, in_channels=3, features=64):
        super().__init__()
        self.initial_down = nn.Sequential(
            nn.Conv2d(in_channels, features, 4, 2, 1, padding_mode="reflect"),
            nn.LeakyReLU(0.2),
        )
        self.down1 = Block(features, features * 2, down=True, act="leaky")
        self.down2 = Block(features * 2, features * 4, down=True, act="leaky")
        self.down3 = Block(features * 4, features * 8, down=True, act="leaky")
        self.down4 = Block(features * 8, features * 8, down=True, act="leaky")
        self.down5 = Block(features * 8, features * 8, down=True, act="leaky")
        self.down6 = Block(features * 8, features * 8, down=True, act="leaky")
        self.bottleneck = nn.Sequential(
            nn.Conv2d(features * 8, features * 8, 4, 2, 1),
            nn.ReLU()
        )
        self.up1 = Block(features * 8, features * 8, down=False, use_dropout=True)
        self.up2 = Block(features * 8 * 2, features * 8, down=False, use_dropout=True)
        self.up3 = Block(features * 8 * 2, features * 8, down=False, use_dropout=True)
        self.up4 = Block(features * 8 * 2, features * 8, down=False)
        self.up5 = Block(features * 8 * 2, features * 4, down=False)
        self.up6 = Block(features * 4 * 2, features * 2, down=False)
        self.up7 = Block(features * 2 * 2, features, down=False)
        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(features * 2, in_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )

    def forward(self, x):
        d1 = self.initial_down(x)
        d2 = self.down1(d1)
        d3 = self.down2(d2)
        d4 = self.down3(d3)
        d5 = self.down4(d4)
        d6 = self.down5(d5)
        d7 = self.down6(d6)
        bottleneck = self.bottleneck(d7)
        up1 = self.up1(bottleneck)
        up2 = self.up2(torch.cat([up1, d7], dim=1))
        up3 = self.up3(torch.cat([up2, d6], dim=1))
        up4 = self.up4(torch.cat([up3, d5], dim=1))
        up5 = self.up5(torch.cat([up4, d4], dim=1))
        up6 = self.up6(torch.cat([up5, d3], dim=1))
        up7 = self.up7(torch.cat([up6, d2], dim=1))
        return self.final_up(torch.cat([up7, d1], dim=1))


Overwriting generator_model.py


In [ ]:
# %%writefile config.py
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Automatically use GPU if available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Google Drive paths (make sure your data is in these folders)
TRAIN_DIR = "/content/drive/MyDrive/output/train"
VAL_DIR = "/content/drive/MyDrive/output/test"

# Hyperparameters
LEARNING_RATE = 2e-4
BATCH_SIZE = 16
NUM_WORKERS = 2
IMAGE_SIZE = 256
CHANNELS_IMG = 3
L1_LAMBDA = 100
LAMBDA_GP = 10
NUM_EPOCHS = 500

# Model checkpoints (inside Colab by default)
LOAD_MODEL = False
SAVE_MODEL = False
CHECKPOINT_DISC = "/content/disc.pth.tar"
CHECKPOINT_GEN = "/content/gen.pth.tar"

# Albumentations transforms
both_transform = A.Compose(
    [A.Resize(width=IMAGE_SIZE, height=IMAGE_SIZE)],
    additional_targets={"image0": "image"},
)

transform_only_input = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(p=0.2),
        A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], max_pixel_value=255.0),
        ToTensorV2(),
    ]
)

transform_only_mask = A.Compose(
    [
        A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], max_pixel_value=255.0),
        ToTensorV2(),
    ]
)


In [ ]:
utils.py
import torch
import config
from torchvision.utils import save_image
import os

def save_some_examples(gen, val_loader, epoch, folder="/content/drive/MyDrive/output/evaluation"):
    os.makedirs(folder, exist_ok=True)
    x, y = next(iter(val_loader))
    x, y = x.to(config.DEVICE), y.to(config.DEVICE)
    gen.eval()
    with torch.no_grad():
        y_fake = gen(x)
        y_fake = y_fake * 0.5 + 0.5  # unnormalize
        save_image(y_fake, os.path.join(folder, f"y_gen_{epoch}.png"))
        save_image(x * 0.5 + 0.5, os.path.join(folder, f"input_{epoch}.png"))
        if epoch == 1:
            save_image(y * 0.5 + 0.5, os.path.join(folder, f"label_{epoch}.png"))
    gen.train()

def save_checkpoint(model, optimizer, filename="/content/drive/MyDrive/output/checkpoints/my_checkpoint.pth.tar"):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    print("=> Saving checkpoint")
    checkpoint = {
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
    }
    torch.save(checkpoint, filename)

def load_checkpoint(checkpoint_file, model, optimizer, lr):
    print("=> Loading checkpoint")
    checkpoint = torch.load(checkpoint_file, map_location=config.DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr


In [ ]:
!pip install albumentations
!pip install --upgrade torch torchvision


KeyboardInterrupt: 

In [ ]:

import os
from torchvision.utils import save_image
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset import MapDataset  # Replace with your dataset import
from generator_model import Generator
from discriminator_model import Discriminator
import config
from utils import save_some_examples, save_checkpoint, load_checkpoint

from google.colab import drive
drive.mount('/content/drive')

EVAL_DIR = "/content/drive/MyDrive/output/evaluation"
os.makedirs(EVAL_DIR, exist_ok=True)

def train_fn(disc, gen, loader, opt_disc, opt_gen, l1, bce, d_scaler, g_scaler):
    loop = tqdm(loader, leave=True)
    for idx, (x, y) in enumerate(loop):
        x = x.to(config.DEVICE)
        y = y.to(config.DEVICE)

        with torch.amp.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
            y_fake = gen(x)
            D_real = disc(x, y)
            D_fake = disc(x, y_fake.detach())
            D_real_loss = bce(D_real, torch.ones_like(D_real))
            D_fake_loss = bce(D_fake, torch.zeros_like(D_fake))
            D_loss = D_real_loss + D_fake_loss

        disc.zero_grad()
        d_scaler.scale(D_loss).backward()
        d_scaler.step(opt_disc)
        d_scaler.update()

        with torch.amp.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
            D_fake = disc(x, y_fake)
            G_fake_loss = bce(D_fake, torch.ones_like(D_fake))
            L1 = l1(y_fake, y) * config.L1_LAMBDA
            G_loss = G_fake_loss + L1

        gen.zero_grad()
        g_scaler.scale(G_loss).backward()
        g_scaler.step(opt_gen)
        g_scaler.update()

        loop.set_description(f"Epoch [{epoch}]")
        loop.set_postfix(D_real=D_real.mean().item(), D_fake=D_fake.mean().item())

def main():
    train_dataset = MapDataset(root_dir=config.TRAIN_DIR)
    val_dataset = MapDataset(root_dir=config.VAL_DIR)
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=1)

    gen = Generator().to(config.DEVICE)
    disc = Discriminator().to(config.DEVICE)
    opt_gen = optim.Adam(gen.parameters(), lr=config.LEARNING_RATE, betas=(0.5, 0.999))
    opt_disc = optim.Adam(disc.parameters(), lr=config.LEARNING_RATE, betas=(0.5, 0.999))
    BCE = nn.BCEWithLogitsLoss()
    L1_LOSS = nn.L1Loss()

    if config.LOAD_MODEL:
        load_checkpoint(config.CHECKPOINT_GEN, gen, opt_gen, config.LEARNING_RATE)
        load_checkpoint(config.CHECKPOINT_DISC, disc, opt_disc, config.LEARNING_RATE)

    g_scaler = torch.amp.GradScaler()
    d_scaler = torch.amp.GradScaler()

    for epoch in range(config.NUM_EPOCHS):
        train_fn(disc, gen, train_loader, opt_disc, opt_gen, L1_LOSS, BCE, d_scaler, g_scaler)

        if config.SAVE_MODEL:
            save_checkpoint(gen, opt_gen, filename=config.CHECKPOINT_GEN)
            save_checkpoint(disc, opt_disc, filename=config.CHECKPOINT_DISC)

        save_some_examples(gen, val_loader, epoch, folder=EVAL_DIR)

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'utils'

In [ ]:
# ✅ FILE: config.py
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LEARNING_RATE = 2e-4
BATCH_SIZE = 16
NUM_WORKERS = 2
IMAGE_SIZE = 256
CHANNELS_IMG = 3
LAMBDA_L1 = 100
LAMBDA_GP = 10
NUM_EPOCHS = 100
LOAD_MODEL = False
SAVE_MODEL = True
CHECKPOINT_GEN = "checkpoints/gen.pth.tar"
CHECKPOINT_DISC = "checkpoints/disc.pth.tar"


# ✅ FILE: generator_model.py
import torch
import torch.nn as nn

def conv_block(in_channels, out_channels, down=True, use_act=True, **kwargs):
    layers = []
    if down:
        layers.append(nn.Conv2d(in_channels, out_channels, padding_mode="reflect", **kwargs))
    else:
        layers.append(nn.ConvTranspose2d(in_channels, out_channels, **kwargs))

    layers.append(nn.BatchNorm2d(out_channels))
    if use_act:
        layers.append(nn.ReLU() if down else nn.LeakyReLU(0.2))
    return nn.Sequential(*layers)

class Generator(nn.Module):
    def __init__(self, in_channels=3, features=64):
        super().__init__()
        self.initial_down = nn.Sequential(
            nn.Conv2d(in_channels, features, kernel_size=4, stride=2, padding=1, padding_mode="reflect"),
            nn.LeakyReLU(0.2),
        )
        self.down1 = conv_block(features, features*2, down=True, kernel_size=4, stride=2, padding=1)
        self.down2 = conv_block(features*2, features*4, down=True, kernel_size=4, stride=2, padding=1)
        self.down3 = conv_block(features*4, features*8, down=True, kernel_size=4, stride=2, padding=1)

        self.bottleneck = nn.Sequential(
            nn.Conv2d(features*8, features*8, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
        )

        self.up1 = conv_block(features*8, features*8, down=False, kernel_size=4, stride=2, padding=1)
        self.up2 = conv_block(features*8*2, features*4, down=False, kernel_size=4, stride=2, padding=1)
        self.up3 = conv_block(features*4*2, features*2, down=False, kernel_size=4, stride=2, padding=1)
        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(features*2*2, in_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )

    def forward(self, x):
        d1 = self.initial_down(x)
        d2 = self.down1(d1)
        d3 = self.down2(d2)
        d4 = self.down3(d3)
        bottleneck = self.bottleneck(d4)
        up1 = self.up1(bottleneck)
        up2 = self.up2(torch.cat([up1, d4], dim=1))
        up3 = self.up3(torch.cat([up2, d3], dim=1))
        return self.final_up(torch.cat([up3, d2], dim=1))


# ✅ FILE: discriminator_model.py
import torch
import torch.nn as nn

class Discriminator(nn.Module):
    def __init__(self, in_channels=3, features=[64, 128, 256, 512]):
        super().__init__()
        layers = []
        self.initial = nn.Sequential(
            nn.Conv2d(in_channels * 2, features[0], kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
        )
        in_channels = features[0]

        for feature in features[1:]:
            layers.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, feature, kernel_size=4, stride=2, padding=1),
                    nn.BatchNorm2d(feature),
                    nn.LeakyReLU(0.2),
                )
            )
            in_channels = feature

        layers.append(
            nn.Conv2d(in_channels, 1, kernel_size=4, stride=1, padding=1)
        )
        self.model = nn.Sequential(*layers)

    def forward(self, x, y):
        x = torch.cat([x, y], dim=1)
        x = self.initial(x)
        return self.model(x)


# ✅ FILE: utils.py
import torch
import config
from torchvision.utils import save_image
import os

def save_some_examples(gen, val_loader, epoch, folder="/content/drive/MyDrive/output/evaluation"):
    os.makedirs(folder, exist_ok=True)
    x, y = next(iter(val_loader))
    x, y = x.to(config.DEVICE), y.to(config.DEVICE)
    gen.eval()
    with torch.no_grad():
        y_fake = gen(x)
        y_fake = y_fake * 0.5 + 0.5
        save_image(y_fake, os.path.join(folder, f"y_gen_{epoch}.png"))
        save_image(x * 0.5 + 0.5, os.path.join(folder, f"input_{epoch}.png"))
        if epoch == 1:
            save_image(y * 0.5 + 0.5, os.path.join(folder, f"label_{epoch}.png"))
    gen.train()

def save_checkpoint(model, optimizer, filename="checkpoints/my_checkpoint.pth.tar"):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    print("=> Saving checkpoint")
    checkpoint = {
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
    }
    torch.save(checkpoint, filename)

def load_checkpoint(checkpoint_file, model, optimizer, lr):
    print("=> Loading checkpoint")
    checkpoint = torch.load(checkpoint_file, map_location=config.DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr


# ✅ FILE: train.py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from generator_model import Generator
from discriminator_model import Discriminator

import config

transform = transforms.Compose([
    transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5 for _ in range(config.CHANNELS_IMG)], [0.5 for _ in range(config.CHANNELS_IMG)])
])

dataset = ImageFolder(root="/content/drive/MyDrive/output/train", transform=transform)
val_dataset = ImageFolder(root="/content/drive/MyDrive/output/test", transform=transform)
train_loader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

gen = Generator().to(config.DEVICE)
disc = Discriminator().to(config.DEVICE)

opt_gen = optim.Adam(gen.parameters(), lr=config.LEARNING_RATE, betas=(0.5, 0.999))
opt_disc = optim.Adam(disc.parameters(), lr=config.LEARNING_RATE, betas=(0.5, 0.999))

bce = nn.BCEWithLogitsLoss()
l1 = nn.L1Loss()

if config.LOAD_MODEL:
    load_checkpoint(config.CHECKPOINT_GEN, gen, opt_gen, config.LEARNING_RATE)
    load_checkpoint(config.CHECKPOINT_DISC, disc, opt_disc, config.LEARNING_RATE)

g_scaler = torch.amp.GradScaler()
d_scaler = torch.amp.GradScaler()


def train_fn():
    loop = enumerate(train_loader)
    for batch_idx, (x, y) in loop:
        x = x.to(config.DEVICE)
        y = y.to(config.DEVICE)

        with torch.amp.autocast(device_type="cuda"):
            y_fake = gen(x)
            D_real = disc(x, y)
            D_fake = disc(x, y_fake.detach())
            D_loss = bce(D_real, torch.ones_like(D_real)) + bce(D_fake, torch.zeros_like(D_fake))

        d_scaler.scale(D_loss).backward()
        d_scaler.step(opt_disc)
        d_scaler.update()
        opt_disc.zero_grad()

        with torch.amp.autocast(device_type="cuda"):
            D_fake = disc(x, y_fake)
            G_loss = bce(D_fake, torch.ones_like(D_fake)) + config.LAMBDA_L1 * l1(y_fake, y)

        g_scaler.scale(G_loss).backward()
        g_scaler.step(opt_gen)
        g_scaler.update()
        opt_gen.zero_grad()

for epoch in range(1, config.NUM_EPOCHS + 1):
    train_fn()
    if config.SAVE_MODEL:
        save_checkpoint(gen, opt_gen, filename=config.CHECKPOINT_GEN)
        save_checkpoint(disc, opt_disc, filename=config.CHECKPOINT_DISC)
    save_some_examples(gen, val_loader, epoch, folder="/content/drive/MyDrive/output/evaluation")


RuntimeError: Tensors must have same number of dimensions: got 4 and 1

In [ ]:


import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
from tqdm import tqdm

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# CONFIG
class Config:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    TRAIN_DIR = "/content/drive/MyDrive/output/train/images"
    VAL_DIR = "/content/drive/MyDrive/output/test/images"
    SAMPLE_DIR = "/content/drive/MyDrive/output/evaluation"
    CHECKPOINT_GEN = "/content/drive/MyDrive/output/checkpoints/gen.pth.tar"
    CHECKPOINT_DISC = "/content/drive/MyDrive/output/checkpoints/disc.pth.tar"
    IMAGE_SIZE = 256
    BATCH_SIZE = 4
    NUM_EPOCHS = 100
    LEARNING_RATE = 2e-4
    NUM_WORKERS = 2
    SAVE_MODEL = True
    L1_LAMBDA = 100

config = Config()
os.makedirs(config.SAMPLE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(config.CHECKPOINT_GEN), exist_ok=True)

# Dataset
class LowLightDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = os.listdir(root_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path = os.path.join(self.root_dir, self.images[index])
        image = Image.open(img_path).convert("RGB")
        w, h = image.size
        input_image = image.crop((0, 0, w//2, h))
        target_image = image.crop((w//2, 0, w, h))

        if self.transform:
            input_image = self.transform(input_image)
            target_image = self.transform(target_image)
        return input_image, target_image

# Transforms
transform = transforms.Compose([
    transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = LowLightDataset(config.TRAIN_DIR, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
val_dataset = LowLightDataset(config.VAL_DIR, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# U-Net Generator
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, down=True, use_dropout=False):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False)
            if down else
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(0.5) if use_dropout else nn.Identity(),
            nn.LeakyReLU(0.2) if down else nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)

class UNetGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.down1 = UNetBlock(3, 64, down=True, use_dropout=False)
        self.down2 = UNetBlock(64, 128, down=True, use_dropout=False)
        self.down3 = UNetBlock(128, 256, down=True, use_dropout=False)

        self.bottleneck = UNetBlock(256, 512, down=True, use_dropout=True)

        self.up1 = UNetBlock(512, 256, down=False, use_dropout=True)
        self.up2 = UNetBlock(512, 128, down=False, use_dropout=False)
        self.up3 = UNetBlock(256, 64, down=False, use_dropout=False)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        bottleneck = self.bottleneck(d3)
        up1 = self.up1(bottleneck)
        up2 = self.up2(torch.cat([up1, d3], dim=1))
        up3 = self.up3(torch.cat([up2, d2], dim=1))
        return self.final(torch.cat([up3, d1], dim=1))

# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(6, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 1, 4, 1, 0),
            nn.Sigmoid()
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], dim=1))

# Initialize
gen = UNetGenerator().to(config.DEVICE)
disc = Discriminator().to(config.DEVICE)
opt_gen = torch.optim.Adam(gen.parameters(), lr=config.LEARNING_RATE, betas=(0.5, 0.999))
opt_disc = torch.optim.Adam(disc.parameters(), lr=config.LEARNING_RATE, betas=(0.5, 0.999))
BCE = nn.BCELoss()
L1 = nn.L1Loss()

# Save Sample
def save_some_examples(gen, loader, epoch):
    x, y = next(iter(loader))
    x = x.to(config.DEVICE)
    gen.eval()
    with torch.no_grad():
        y_fake = gen(x)
        y_fake = y_fake * 0.5 + 0.5
        x = x * 0.5 + 0.5
    save_image(y_fake, os.path.join(config.SAMPLE_DIR, f"y_gen_{epoch}.png"))
    save_image(x, os.path.join(config.SAMPLE_DIR, f"input_{epoch}.png"))
    gen.train()

# Training Loop
for epoch in range(config.NUM_EPOCHS):
    loop = tqdm(train_loader, leave=True)
    for idx, (x, y) in enumerate(loop):
        x, y = x.to(config.DEVICE), y.to(config.DEVICE)

        # Train Discriminator
        y_fake = gen(x)
        D_real = disc(x, y)
        D_fake = disc(x, y_fake.detach())
        D_loss = BCE(D_real, torch.ones_like(D_real)) + BCE(D_fake, torch.zeros_like(D_fake))
        disc.zero_grad()
        D_loss.backward()
        opt_disc.step()

        # Train Generator
        D_fake = disc(x, y_fake)
        G_loss = BCE(D_fake, torch.ones_like(D_fake)) + config.L1_LAMBDA * L1(y_fake, y)
        gen.zero_grad()
        G_loss.backward()
        opt_gen.step()

        loop.set_description(f"Epoch [{epoch+1}/{config.NUM_EPOCHS}]")
        loop.set_postfix(D_loss=D_loss.item(), G_loss=G_loss.item())

    if config.SAVE_MODEL:
        torch.save(gen.state_dict(), config.CHECKPOINT_GEN)
        torch.save(disc.state_dict(), config.CHECKPOINT_DISC)

    save_some_examples(gen, val_loader, epoch+1)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Epoch [100/100]: 100%|██████████| 513/513 [01:26<00:00,  5.90it/s, D_loss=0.716, G_loss=5.68]


In [ ]:
# GAN for Low-light Enhancement using Side-by-Side Images (Colab Friendly) with U-Net Generator and MiDaS Depth Estimation
# Folder structure:
# /content/drive/MyDrive/output/train/image/*.png (side-by-side: lowlight | normal)
# /content/drive/MyDrive/output/test/image/*.png

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import cv2
import urllib.request

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# CONFIG
class Config:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    TRAIN_DIR = "/content/drive/MyDrive/output/train/images"
    VAL_DIR = "/content/drive/MyDrive/output/test/images"
    SAMPLE_DIR = "/content/drive/MyDrive/output/evaluation"
    CHECKPOINT_GEN = "/content/drive/MyDrive/output/checkpoints/gen.pth.tar"
    CHECKPOINT_DISC = "/content/drive/MyDrive/output/checkpoints/disc.pth.tar"
    IMAGE_SIZE = 256
    BATCH_SIZE = 4
    NUM_EPOCHS = 10
    LEARNING_RATE = 2e-4
    NUM_WORKERS = 2
    SAVE_MODEL = True
    L1_LAMBDA = 100

config = Config()
os.makedirs(config.SAMPLE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(config.CHECKPOINT_GEN), exist_ok=True)

# Dataset
class LowLightDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = os.listdir(root_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path = os.path.join(self.root_dir, self.images[index])
        image = Image.open(img_path).convert("RGB")
        w, h = image.size
        input_image = image.crop((0, 0, w//2, h))
        target_image = image.crop((w//2, 0, w, h))

        if self.transform:
            input_image = self.transform(input_image)
            target_image = self.transform(target_image)
        return input_image, target_image

# Transforms
transform = transforms.Compose([
    transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = LowLightDataset(config.TRAIN_DIR, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
val_dataset = LowLightDataset(config.VAL_DIR, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# U-Net Generator
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, down=True, use_dropout=False):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False)
            if down else
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(0.5) if use_dropout else nn.Identity(),
            nn.LeakyReLU(0.2) if down else nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)

class UNetGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.down1 = UNetBlock(3, 64, down=True, use_dropout=False)
        self.down2 = UNetBlock(64, 128, down=True, use_dropout=False)
        self.down3 = UNetBlock(128, 256, down=True, use_dropout=False)

        self.bottleneck = UNetBlock(256, 512, down=True, use_dropout=True)

        self.up1 = UNetBlock(512, 256, down=False, use_dropout=True)
        self.up2 = UNetBlock(512, 128, down=False, use_dropout=False)
        self.up3 = UNetBlock(256, 64, down=False, use_dropout=False)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        bottleneck = self.bottleneck(d3)
        up1 = self.up1(bottleneck)
        up2 = self.up2(torch.cat([up1, d3], dim=1))
        up3 = self.up3(torch.cat([up2, d2], dim=1))
        return self.final(torch.cat([up3, d1], dim=1))

# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(6, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 1, 4, 1, 0),
            nn.Sigmoid()
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], dim=1))

# Initialize
gen = UNetGenerator().to(config.DEVICE)
disc = Discriminator().to(config.DEVICE)
opt_gen = torch.optim.Adam(gen.parameters(), lr=config.LEARNING_RATE, betas=(0.5, 0.999))
opt_disc = torch.optim.Adam(disc.parameters(), lr=config.LEARNING_RATE, betas=(0.5, 0.999))
BCE = nn.BCELoss()
L1 = nn.L1Loss()

# MiDaS Setup (Lightweight Model)
def setup_midas():
    midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small").to(config.DEVICE)
    midas.eval()
    transform = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform
    return midas, transform

midas_model, midas_transform = setup_midas()

def estimate_depth(image_tensor):
    input_batch = midas_transform(image_tensor.squeeze().cpu()).to(config.DEVICE)
    with torch.no_grad():
        prediction = midas_model(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=(config.IMAGE_SIZE, config.IMAGE_SIZE),
            mode="bicubic",
            align_corners=False,
        ).squeeze()
    return prediction

# Save Sample + Depth
def save_some_examples(gen, loader, epoch):
    x, y = next(iter(loader))
    x = x.to(config.DEVICE)
    gen.eval()
    with torch.no_grad():
        y_fake = gen(x)
        y_fake_vis = y_fake * 0.5 + 0.5
        x_vis = x * 0.5 + 0.5
        save_image(y_fake_vis, os.path.join(config.SAMPLE_DIR, f"y_gen_{epoch}.png"))
        save_image(x_vis, os.path.join(config.SAMPLE_DIR, f"input_{epoch}.png"))

        # Save depth map from generated image
        depth = estimate_depth(y_fake_vis)
        plt.imsave(os.path.join(config.SAMPLE_DIR, f"depth_{epoch}.png"), depth.cpu(), cmap='inferno')

    gen.train()

# Training Loop
for epoch in range(config.NUM_EPOCHS):
    loop = tqdm(train_loader, leave=True)
    for idx, (x, y) in enumerate(loop):
        x, y = x.to(config.DEVICE), y.to(config.DEVICE)

        # Train Discriminator
        y_fake = gen(x)
        D_real = disc(x, y)
        D_fake = disc(x, y_fake.detach())
        D_loss = BCE(D_real, torch.ones_like(D_real)) + BCE(D_fake, torch.zeros_like(D_fake))
        disc.zero_grad()
        D_loss.backward()
        opt_disc.step()

        # Train Generator
        D_fake = disc(x, y_fake)
        G_loss = BCE(D_fake, torch.ones_like(D_fake)) + config.L1_LAMBDA * L1(y_fake, y)
        gen.zero_grad()
        G_loss.backward()
        opt_gen.step()

        loop.set_description(f"Epoch [{epoch+1}/{config.NUM_EPOCHS}]")
        loop.set_postfix(D_loss=D_loss.item(), G_loss=G_loss.item())

    if config.SAVE_MODEL:
        torch.save(gen.state_dict(), config.CHECKPOINT_GEN)
        torch.save(disc.state_dict(), config.CHECKPOINT_DISC)

    save_some_examples(gen, val_loader, epoch+1)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.11/dist-packages/torch/hub.py:330: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip
/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.l

Loading weights:  None


/usr/local/lib/python3.11/dist-packages/torch/hub.py:330: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_lite3-b733e338.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientnet_lite3-b733e338.pth
Downloading: "https://github.

KeyboardInterrupt: 

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
from tqdm import tqdm

# -------------------
# CONFIG
# -------------------
class Config:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    TRAIN_DIR = "/content/drive/MyDrive/output/train/images"
    VAL_DIR = "/content/drive/MyDrive/output/test/images"
    SAMPLE_DIR = "/content/drive/MyDrive/output/evaluation"
    CHECKPOINT_GEN = "/content/drive/MyDrive/output/checkpoints/gen_resnet.pth.tar"
    CHECKPOINT_DISC = "/content/drive/MyDrive/output/checkpoints/disc_resnet.pth.tar"
    IMAGE_SIZE = 256
    BATCH_SIZE = 4
    NUM_EPOCHS = 100
    LEARNING_RATE = 2e-4
    NUM_WORKERS = 2
    SAVE_MODEL = True
    L1_LAMBDA = 100

config = Config()
os.makedirs(config.SAMPLE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(config.CHECKPOINT_GEN), exist_ok=True)

# -------------------
# Dataset
# -------------------
class LowLightDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = os.listdir(root_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path = os.path.join(self.root_dir, self.images[index])
        image = Image.open(img_path).convert("RGB")
        w, h = image.size
        # Left = input (low light), Right = target (enhanced)
        input_image = image.crop((0, 0, w//2, h))
        target_image = image.crop((w//2, 0, w, h))

        if self.transform:
            input_image = self.transform(input_image)
            target_image = self.transform(target_image)
        return input_image, target_image

# -------------------
# Transforms
# -------------------
transform = transforms.Compose([
    transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = LowLightDataset(config.TRAIN_DIR, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
val_dataset = LowLightDataset(config.VAL_DIR, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# -------------------
# Residual Block
# -------------------
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)


class ResNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, n_residual_blocks=6):
        super().__init__()

        # Initial convolution (keeps 256x256)
        model = [
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        ]

        # Downsampling -> 64x64
        in_features = 64
        out_features = in_features * 2
        for _ in range(2):
            model += [
                nn.Conv2d(in_features, out_features, kernel_size=3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features * 2

        # Residual blocks (bottleneck at 64x64)
        for _ in range(n_residual_blocks):
            model += [ResidualBlock(in_features)]

        # Upsampling -> back to 256x256
        out_features = in_features // 2
        for _ in range(2):
            model += [
                nn.ConvTranspose2d(in_features, out_features, kernel_size=3, stride=2,
                                   padding=1, output_padding=1, bias=False),
                nn.BatchNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features // 2

        # Output layer
        model += [
            nn.Conv2d(64, out_channels, kernel_size=7, stride=1, padding=3),
            nn.Tanh()
        ]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


# -------------------
# PatchGAN Discriminator
# -------------------
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(6, 64, 4, 2, 1),  # input: real+fake concat
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1, 1),  # PatchGAN output
            nn.Sigmoid()
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], dim=1))

# -------------------
# Initialize
# -------------------
gen = ResNetGenerator().to(config.DEVICE)
disc = Discriminator().to(config.DEVICE)
opt_gen = torch.optim.Adam(gen.parameters(), lr=config.LEARNING_RATE, betas=(0.5, 0.999))
opt_disc = torch.optim.Adam(disc.parameters(), lr=config.LEARNING_RATE, betas=(0.5, 0.999))
BCE = nn.BCELoss()
L1 = nn.L1Loss()

# -------------------
# Save Sample
# -------------------
def save_some_examples(gen, loader, epoch):
    x, y = next(iter(loader))
    x = x.to(config.DEVICE)
    gen.eval()
    with torch.no_grad():
        y_fake = gen(x)
        y_fake = y_fake * 0.5 + 0.5
        x = x * 0.5 + 0.5
    save_image(y_fake, os.path.join(config.SAMPLE_DIR, f"y_gen_{epoch}.png"))
    save_image(x, os.path.join(config.SAMPLE_DIR, f"input_{epoch}.png"))
    gen.train()

# -------------------
# Training Loop
# -------------------
def train():
    for epoch in range(config.NUM_EPOCHS):
        loop = tqdm(train_loader, leave=True)
        for idx, (x, y) in enumerate(loop):
            x, y = x.to(config.DEVICE), y.to(config.DEVICE)

            # Train Discriminator
            y_fake = gen(x)
            D_real = disc(x, y)
            D_fake = disc(x, y_fake.detach())
            D_loss = BCE(D_real, torch.ones_like(D_real)) + BCE(D_fake, torch.zeros_like(D_fake))
            disc.zero_grad()
            D_loss.backward()
            opt_disc.step()

            # Train Generator
            D_fake = disc(x, y_fake)
            G_loss = BCE(D_fake, torch.ones_like(D_fake)) + config.L1_LAMBDA * L1(y_fake, y)
            gen.zero_grad()
            G_loss.backward()
            opt_gen.step()

            loop.set_description(f"Epoch [{epoch+1}/{config.NUM_EPOCHS}]")
            loop.set_postfix(D_loss=D_loss.item(), G_loss=G_loss.item())

        if config.SAVE_MODEL:
            torch.save(gen.state_dict(), config.CHECKPOINT_GEN)
            torch.save(disc.state_dict(), config.CHECKPOINT_DISC)

        save_some_examples(gen, val_loader, epoch+1)

# -------------------
# Inference (Enhance Single Image)
# -------------------
def enhance_image(image_path, save_path="enhanced.png"):
    gen = ResNetGenerator().to(config.DEVICE)
    gen.load_state_dict(torch.load(config.CHECKPOINT_GEN, map_location=config.DEVICE))
    gen.eval()

    transform_infer = transforms.Compose([
        transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    image = Image.open(image_path).convert("RGB")
    input_tensor = transform_infer(image).unsqueeze(0).to(config.DEVICE)

    with torch.no_grad():
        fake = gen(input_tensor)
        fake = fake * 0.5 + 0.5  # De-normalize to [0,1]

    save_image(fake, save_path)
    print(f"Enhanced image saved at {save_path}")

# -------------------
# Run Training (Uncomment to Train)
# -------------------
# train()

# -------------------
# Example Inference (After Training)
# -------------------
# enhance_image("/content/drive/MyDrive/test_lowlight.png", save_path="result.png")


In [ ]:
enhance_image(
    image_path="/content/2.png",
    save_path="/content/result.png"
)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/output/checkpoints/gen_resnet.pth.tar'

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)


class ResNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, n_residual_blocks=6):
        super().__init__()

        # Initial convolution (keeps 256x256)
        model = [
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        ]

        # Downsampling -> 64x64
        in_features = 64
        out_features = in_features * 2
        for _ in range(2):
            model += [
                nn.Conv2d(in_features, out_features, kernel_size=3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features * 2

        # Residual blocks (bottleneck at 64x64)
        for _ in range(n_residual_blocks):
            model += [ResidualBlock(in_features)]

        # Upsampling -> back to 256x256
        out_features = in_features // 2
        for _ in range(2):
            model += [
                nn.ConvTranspose2d(in_features, out_features, kernel_size=3, stride=2,
                                   padding=1, output_padding=1, bias=False),
                nn.BatchNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features // 2

        # Output layer
        model += [
            nn.Conv2d(64, out_channels, kernel_size=7, stride=1, padding=3),
            nn.Tanh()
        ]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
from tqdm import tqdm

# -------------------
# Mount Google Drive
# -------------------
from google.colab import drive
drive.mount('/content/drive')

# -------------------
# CONFIG
# -------------------
class Config:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    TRAIN_DIR = "/content/drive/MyDrive/output/train/images"
    VAL_DIR = "/content/drive/MyDrive/output/test/images"
    SAMPLE_DIR = "/content/drive/MyDrive/output/evaluation"
    CHECKPOINT_GEN = "/content/drive/MyDrive/output/checkpoints/gen.pth.tar"
    CHECKPOINT_DISC = "/content/drive/MyDrive/output/checkpoints/disc.pth.tar"
    IMAGE_SIZE = 256
    BATCH_SIZE = 4
    NUM_EPOCHS = 100
    LEARNING_RATE = 2e-4
    NUM_WORKERS = 2
    SAVE_MODEL = True
    L1_LAMBDA = 100

config = Config()
os.makedirs(config.SAMPLE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(config.CHECKPOINT_GEN), exist_ok=True)

# -------------------
# Dataset
# -------------------
class LowLightDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = os.listdir(root_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path = os.path.join(self.root_dir, self.images[index])
        image = Image.open(img_path).convert("RGB")
        w, h = image.size

        # Left = low-light, Right = enhanced
        input_image = image.crop((0, 0, w//2, h))
        target_image = image.crop((w//2, 0, w, h))

        if self.transform:
            input_image = self.transform(input_image)
            target_image = self.transform(target_image)
        return input_image, target_image

# -------------------
# Transforms
# -------------------
transform = transforms.Compose([
    transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = LowLightDataset(config.TRAIN_DIR, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
val_dataset = LowLightDataset(config.VAL_DIR, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# -------------------
# U-Net Generator
# -------------------
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, down=True, use_dropout=False):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False)
            if down else
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(0.5) if use_dropout else nn.Identity(),
            nn.LeakyReLU(0.2) if down else nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)

class UNetGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.down1 = UNetBlock(3, 64, down=True, use_dropout=False)
        self.down2 = UNetBlock(64, 128, down=True, use_dropout=False)
        self.down3 = UNetBlock(128, 256, down=True, use_dropout=False)

        self.bottleneck = UNetBlock(256, 512, down=True, use_dropout=True)

        self.up1 = UNetBlock(512, 256, down=False, use_dropout=True)
        self.up2 = UNetBlock(512, 128, down=False, use_dropout=False)
        self.up3 = UNetBlock(256, 64, down=False, use_dropout=False)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        bottleneck = self.bottleneck(d3)
        up1 = self.up1(bottleneck)
        up2 = self.up2(torch.cat([up1, d3], dim=1))
        up3 = self.up3(torch.cat([up2, d2], dim=1))
        return self.final(torch.cat([up3, d1], dim=1))

# -------------------
# Discriminator
# -------------------
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(6, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 1, 4, 1, 0),
            nn.Sigmoid()
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], dim=1))

# -------------------
# Initialize Models + Optimizers
# -------------------
gen = UNetGenerator().to(config.DEVICE)
disc = Discriminator().to(config.DEVICE)
opt_gen = torch.optim.Adam(gen.parameters(), lr=config.LEARNING_RATE, betas=(0.5, 0.999))
opt_disc = torch.optim.Adam(disc.parameters(), lr=config.LEARNING_RATE, betas=(0.5, 0.999))
BCE = nn.BCELoss()
L1 = nn.L1Loss()

# -------------------
# Save Sample Images
# -------------------
def save_some_examples(gen, loader, epoch):
    x, y = next(iter(loader))
    x = x.to(config.DEVICE)
    gen.eval()
    with torch.no_grad():
        y_fake = gen(x)
        y_fake = y_fake * 0.5 + 0.5
        x = x * 0.5 + 0.5
    save_image(y_fake, os.path.join(config.SAMPLE_DIR, f"y_gen_{epoch}.png"))
    save_image(x, os.path.join(config.SAMPLE_DIR, f"input_{epoch}.png"))
    gen.train()

# -------------------
# Training Loop
# -------------------
def train():
    for epoch in range(config.NUM_EPOCHS):
        loop = tqdm(train_loader, leave=True)
        for idx, (x, y) in enumerate(loop):
            x, y = x.to(config.DEVICE), y.to(config.DEVICE)

            # Train Discriminator
            y_fake = gen(x)
            D_real = disc(x, y)
            D_fake = disc(x, y_fake.detach())
            D_loss = BCE(D_real, torch.ones_like(D_real)) + BCE(D_fake, torch.zeros_like(D_fake))
            disc.zero_grad()
            D_loss.backward()
            opt_disc.step()

            # Train Generator
            D_fake = disc(x, y_fake)
            G_loss = BCE(D_fake, torch.ones_like(D_fake)) + config.L1_LAMBDA * L1(y_fake, y)
            gen.zero_grad()
            G_loss.backward()
            opt_gen.step()

            loop.set_description(f"Epoch [{epoch+1}/{config.NUM_EPOCHS}]")
            loop.set_postfix(D_loss=D_loss.item(), G_loss=G_loss.item())

        if config.SAVE_MODEL:
            torch.save(gen.state_dict(), config.CHECKPOINT_GEN)
            torch.save(disc.state_dict(), config.CHECKPOINT_DISC)

        save_some_examples(gen, val_loader, epoch+1)

# -------------------
# Inference (Enhance Single Image)
# -------------------
def enhance_image(image_path, save_path="/content/drive/MyDrive/output/evaluation/enhanced.png"):
    # Load trained generator
    gen = UNetGenerator().to(config.DEVICE)
    gen.load_state_dict(torch.load(config.CHECKPOINT_GEN, map_location=config.DEVICE))
    gen.eval()

    transform_infer = transforms.Compose([
        transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    # Load image
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform_infer(image).unsqueeze(0).to(config.DEVICE)

    # Generate enhanced output
    with torch.no_grad():
        fake = gen(input_tensor)
        fake = fake * 0.5 + 0.5  # de-normalize to [0,1]

    save_image(fake, save_path)
    print(f"Enhanced image saved at {save_path}")

# -------------------
# Run Training
# -------------------
train()

# -------------------
# Example Inference
# -------------------
# enhance_image("/content/drive/MyDrive/test_lowlight.png",
#               save_path="/content/drive/MyDrive/output/evaluation/my_result.png")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Epoch [100/100]: 100%|██████████| 513/513 [01:25<00:00,  5.98it/s, D_loss=0.3, G_loss=5.35]


In [ ]:
enhance_image("/content/273.png",
             save_path="/content/res7.png")

Enhanced image saved at /content/res7.png
